In [1]:
import pickle
import os
import time
import numpy as np
import random

In [2]:
runtime = time.time()
home = './datasets'

In [3]:
DATASET_DIR = home + '/amazon'
#ORDERS_FILE = DATASET_DIR + '/Clothing_Shoes_and_Jewelry_Result_sample.csv'
ORDERS_FILE = DATASET_DIR + '/Clothing_Shoes_and_Jewelry_Result_2012_2014.csv'

'''
DATASET_USER_SESSIONS_SORTING = DATASET_DIR + '/user_sessions_sorting.pickle'
DATASET_USER_SESSIONS_RENAME = DATASET_DIR + '/user_sessions_rename.pickle'
DATASET_TRAIN_TEST_SPLIT_PAD_VALUE = DATASET_DIR + '/train_test_split_pad_value.pickle'
'''
DATASET_USER_SESSIONS_SORTING = DATASET_DIR + '/2_user_sessions.pickle'
DATASET_USER_SESSIONS_RENAME = DATASET_DIR + '/3_user_sessions_filter.pickle'
DATASET_TRAIN_TEST_SPLIT_PAD_VALUE = DATASET_DIR + '/4_train_test_split.pickle'
DATASET_TRAIN_TEST_SPLIT_PAD_VALUE_SAMPLE = DATASET_DIR + '/4_train_test_split_sample.pickle'

In [4]:
MAX_SESSION_LENGTH = 20
MINIMUM_REQUIRED_SESSIONS = 3 #如果用戶的session中的互動少於X，則刪過少的session
PAD_VALUE = 0 #補值:補0

In [5]:
def file_exists(filename):
    return os.path.isfile(filename)

def load_pickle(pickle_file):
    return pickle.load(open(pickle_file, 'rb'))

def save_pickle(data_object, data_file):
    pickle.dump(data_object, open(data_file, 'wb'))

# user_sessions_sorting

In [6]:
if not file_exists(DATASET_USER_SESSIONS_SORTING):
    user_orders = {}
    print("Reading user orders.")
    
    with open(ORDERS_FILE, 'rt', buffering=10000, encoding='utf8') as orders:
        next(orders) # skip header
        # user_id  product_id  reviewTime  unixReviewTime
        for line in orders:
            line = line.rstrip() #刪除空行
            line = line.split(',') #將每行遇到,就用''切割成value
            
            #各自賦予意義: 用戶_id / 產品_id / 瀏覽時長(原始) / 瀏覽時長(nuix)
            user_id        = line[0] 
            product_id     = line[1]
            reviewTime     = line[2]+line[3]
            unixReviewTime = line[4]
            
            #判斷 如果用戶_id不在裡面就...
            if user_id not in user_orders:
                
                #在此uesr_id的位置設立空集合
                user_orders[user_id] = [] 
                
            #將方才的2個value放入一個[]append進去，此時裡面是2維向量   
            user_orders[user_id].append([product_id, unixReviewTime]) 
            
            #這筆資料集有402093個用戶，當我key超過閥值就break，但顯然是不會超過
            if len(user_orders.keys()) >1000000 : 
                break

Reading user orders.


In [7]:
print("Sorting user orders.")
# ensure that orders are sorted for each user

for user_id in user_orders.keys():
    orders = user_orders[user_id]
    user_orders[user_id] = sorted(orders, key=lambda x: x[1]) #利用key，按照時間長短x[1]排序

Sorting user orders.


In [8]:
print("Connecting orders and users.")

results={}
results_orders={}

for user_id in user_orders.keys():
    results[user_id] = {}
    for x in user_orders[user_id]:
        if x[1] not in results[user_id]:#x[1]點擊時間
            results[user_id][x[1]]=[]

        results[user_id][x[1]].append([len(results[user_id][x[1]])+1,x[0]])


for user_id in results.keys():
    results_orders[user_id] = []
    for unixReviewTime in results[user_id].keys():
        results_orders[user_id].append(results[user_id][unixReviewTime])
    #print(results_orders[user_id])
    user_orders[user_id]=results_orders[user_id]


del results
del results_orders

Connecting orders and users.


In [9]:
print("Calculating some statistics.")
session_lengths = [0]*155
products = {}
n_sessions = 0
longest = 0
shortest = 999999
for user, orders in user_orders.items():
    n_sessions += len(orders) #計算session有幾個? 直接計算orders多長就好
    for order in orders:
        if len(order) > longest:
            longest = len(order)  #當滿足條件 就會一直被更新最長的session
        if len(order) < shortest:
            shortest = len(order) #當滿足條件 就會一直被更新最短的session
        session_lengths[len(order)] += 1 #蠻特殊的，每個session中內容多長，用len計算，並將之+1記錄到session_lengths[]符合的位置
        for x in order:
            product = x[1]
            if product not in products:
                products[product] = True #只是將與之對應位置賦予True
print("num products (labels):", len(products.keys())) #幾種商品
print("num users:", len(user_orders.keys())) #幾個user
print("num sessions:", n_sessions) #幾個session
print("shortest session:", shortest) #經累計計算後最短session
print("longest session:", longest) #經累計計算後最長session
print()
print("SESSION LENGTHS:")
for i in range(len(session_lengths)):
    print(i, session_lengths[i]) #將方才紀錄的位置show出來

save_pickle(user_orders, DATASET_USER_SESSIONS_SORTING)

Calculating some statistics.
num products (labels): 140116
num users: 402093
num sessions: 770290
shortest session: 1
longest session: 60

SESSION LENGTHS:
0 0
1 516990
2 140103
3 53164
4 25299
5 14738
6 7324
7 3375
8 1813
9 1784
10 1401
11 1362
12 471
13 287
14 269
15 242
16 130
17 150
18 163
19 351
20 284
21 187
22 87
23 48
24 43
25 22
26 14
27 16
28 36
29 37
30 29
31 24
32 12
33 8
34 7
35 3
36 5
37 1
38 3
39 1
40 2
41 0
42 1
43 0
44 1
45 0
46 0
47 0
48 2
49 0
50 0
51 0
52 0
53 0
54 0
55 0
56 0
57 0
58 0
59 0
60 1
61 0
62 0
63 0
64 0
65 0
66 0
67 0
68 0
69 0
70 0
71 0
72 0
73 0
74 0
75 0
76 0
77 0
78 0
79 0
80 0
81 0
82 0
83 0
84 0
85 0
86 0
87 0
88 0
89 0
90 0
91 0
92 0
93 0
94 0
95 0
96 0
97 0
98 0
99 0
100 0
101 0
102 0
103 0
104 0
105 0
106 0
107 0
108 0
109 0
110 0
111 0
112 0
113 0
114 0
115 0
116 0
117 0
118 0
119 0
120 0
121 0
122 0
123 0
124 0
125 0
126 0
127 0
128 0
129 0
130 0
131 0
132 0
133 0
134 0
135 0
136 0
137 0
138 0
139 0
140 0
141 0
142 0
143 0
144 0
145 0
146 0
1

# user_sessions_rename

In [10]:
if not file_exists(DATASET_USER_SESSIONS_RENAME):
    user_sessions = load_pickle(DATASET_USER_SESSIONS_SORTING)

    # Split sessions
    def split_single_session(session):  
        splitted = [session[i:i+MAX_SESSION_LENGTH] for i in range(0, len(session), MAX_SESSION_LENGTH)]
        if len(splitted[-1]) < 2:
            del splitted[-1]
        return splitted

    def perform_session_splits(sessions):
        splitted_sessions = []
        for session in sessions:
            splitted_sessions += split_single_session(session)
        return splitted_sessions

In [11]:
#print(user_sessions)
for k in user_sessions.keys():
    sessions = user_sessions[k]
    #print(sessions)
    user_sessions[k] = perform_session_splits(sessions)
    #print(user_sessions[k])

In [12]:
# Remove too short sessions 刪掉過短session
for k in user_sessions.keys():
    sessions = user_sessions[k]
    user_sessions[k] = [s for s in sessions if len(s)>1]
    #print(user_sessions[k])

In [13]:
# Remove users with too few sessions 刪掉過少的session
users_to_remove = []
for u, sessions in user_sessions.items():
    if len(sessions) < MINIMUM_REQUIRED_SESSIONS:
        users_to_remove.append(u)

for u in users_to_remove:
    del(user_sessions[u])
    
#for k in user_sessions.items():
    #print(k)

In [14]:
# Rename user_ids 用戶
if len(users_to_remove) > 0:
    nus = {}
    for k, v in user_sessions.items():
        nus[len(nus)] = user_sessions[k] 
#nus

In [15]:
# Rename labels
lab = {}
for k, v in nus.items():
    for session in v:
        for i in range(len(session)):
            if isinstance(session[i][1], str) == True:
                l = session[i][1]
                if l not in lab:
                    lab[l] = len(lab)+1
                session[i][1] = lab[l]
#nus

In [16]:
print('檢查是否有空值')
for k, v in nus.items():
    for session in v:
        if not session:
            print('Has Empty !!!!!'+ str(session))

save_pickle(nus, DATASET_USER_SESSIONS_RENAME)

檢查是否有空值


# train_test_split_pad_value

In [17]:
def get_session_lengths(dataset):
    session_lengths = {}
    for k, v in dataset.items():
        session_lengths[k] = []
        for session in v:
            session_lengths[k].append(len(session)-1)
    return session_lengths

def create_padded_sequence(session):
    if len(session) == MAX_SESSION_LENGTH:
        return session

    length_to_pad = MAX_SESSION_LENGTH - len(session)
    padding = [[PAD_VALUE, PAD_VALUE]] * length_to_pad
    session += padding
    return session

def pad_sequences(dataset):
    for k, v in dataset.items():
        for session_index in range(len(v)):
            dataset[k][session_index] = create_padded_sequence(dataset[k][session_index])

# sample

In [18]:
#sample data
if not file_exists(DATASET_TRAIN_TEST_SPLIT_PAD_VALUE_SAMPLE):
    dataset = load_pickle(DATASET_USER_SESSIONS_RENAME)
    trainset = {}
    testset  = {}

#sample data
session_lengths_1 = {}
for k, v in dataset.items():
    session_lengths_1[k] = []
    for session in v:
        session_lengths_1[k].append(len(session)-1)
        
#sample data
count = 0
for k, v in dataset.items():
    
    count += 1
    if count == 100:   #取到100位user(但會算0，所以key只有到99)
        break
            
    n_sessions_1 = len(v)
    split_point = int(0.8 * n_sessions_1) #80%去切sessions

    if split_point < 2: #只是再次確認是不是有過短的session
        raise ValueError('WTF? so few sessions?')

    trainset[k] = v[:split_point] #例如:v[1,2,3,4]，[:2]，代表V的一開始到2 = [1,2]
    testset[k] = v[split_point:] #代表v的2後一位到最後一個 = [3,4]

# Also need to know the session lengths
train_session_lengths = get_session_lengths(trainset) #統計長度
test_session_lengths = get_session_lengths(testset)

# Finally, pad all sequences before storing everything
pad_sequences(trainset)
pad_sequences(testset)

# Put everything in a dict, and just store the dict with pickle
pickle_dict = {}
pickle_dict['trainset'] = trainset
pickle_dict['testset'] = testset
pickle_dict['train_session_lengths'] = train_session_lengths
pickle_dict['test_session_lengths'] = test_session_lengths
save_pickle(pickle_dict , DATASET_TRAIN_TEST_SPLIT_PAD_VALUE_SAMPLE)

# Fully

In [19]:
if not file_exists(DATASET_TRAIN_TEST_SPLIT_PAD_VALUE):
    dataset = load_pickle(DATASET_USER_SESSIONS_RENAME)
    trainset = {}
    testset  = {}
    session_lengths = {}
    for k, v in dataset.items():
        session_lengths[k] = []
        for session in v:
            session_lengths[k].append(len(session)-1)
    #print(session_lengths)

    for k, v in dataset.items():
        n_sessions = len(v)
        split_point = int(0.8 * n_sessions) #80%去切sessions

        if split_point < 2: #只是再次確認是不是有過短的session
            raise ValueError('WTF? so few sessions?')

        trainset[k] = v[:split_point] #例如:v[1,2,3,4]，[:2]，代表V的一開始到2 = [1,2]
        testset[k] = v[split_point:] #代表v的2後一位到最後一個 = [3,4]

    # Also need to know the session lengths
    train_session_lengths = get_session_lengths(trainset) #統計長度
    test_session_lengths = get_session_lengths(testset)

    # Finally, pad all sequences before storing everything
    pad_sequences(trainset)
    pad_sequences(testset)

    # Put everything in a dict, and just store the dict with pickle
    pickle_dict = {}
    #print(pickle_dict)
    pickle_dict['trainset'] = trainset
    #print(trainset)
    #print(pickle_dict['trainset'])
    pickle_dict['testset'] = testset
    #print(type(pickle_dict['testset']))
    #print(pickle_dict['testset'])
    pickle_dict['train_session_lengths'] = train_session_lengths
    #print(train_session_lengths)
    pickle_dict['test_session_lengths'] = test_session_lengths
    #print(test_session_lengths)
    save_pickle(pickle_dict , DATASET_TRAIN_TEST_SPLIT_PAD_VALUE)
    print("Calculating some statistics.")
    session_lengths = [0]*22
    products = {}
    n_sessions = 0
    longest = 0
    shortest = 999999

Calculating some statistics.


# 統計數據

In [20]:
for user, orders in dataset.items():
    n_sessions += len(orders)
    for order in orders:
        if len(order) > longest:
            longest = len(order)
        if len(order) < shortest:
            shortest = len(order)
        session_lengths[len(order)] += 1
        for x in order:
            product = x[1]
            if product not in products:
                products[product] = True
print("num products (labels):", len(products.keys()))
print("num users:", len(dataset.keys()))
print("num sessions:", n_sessions)
print("shortest session:", shortest)
print("longest session:", longest)
print()
print("SESSION LENGTHS:")
for i in range(len(session_lengths)):
    print(i, session_lengths[i])
print("runtime:", str(time.time() - runtime))

num products (labels): 46959
num users: 9733
num sessions: 35048
shortest session: 20
longest session: 20

SESSION LENGTHS:
0 0
1 0
2 0
3 0
4 0
5 0
6 0
7 0
8 0
9 0
10 0
11 0
12 0
13 0
14 0
15 0
16 0
17 0
18 0
19 0
20 35048
21 0
runtime: 22.520692110061646
